In [1]:
import sys
import os
import torch

# 1. 确保项目根目录在路径中
# 假设您的 Notebook 在 OAReactDiff-LYY 根目录下
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)

# 2. 使用新的路径导入
try:
    # 注意：现在路径变了，要加上 oa_reactdiff 前缀
    from oa_reactdiff.dataset.transition1x import ProcessedTS1x
    print("✅ 导入成功！")
except ImportError as e:
    print(f"❌ 导入失败: {e}")
    print("请检查 oa_reactdiff/dataset/transition1x.py 里的 import 语句是否已修改为 'from .base_dataset ...'")

✅ 导入成功！


In [9]:
import numpy as np
from pathlib import Path
import os

# 假设您的 NPZ 文件路径
NPZ_FILE_PATH = Path("./oa_reactdiff/data_meci/custom_R_P_multi_molecule.npz") 

def analyze_custom_npz(file_path: Path):
    """加载 NPZ 文件并打印样本数和文件大小。"""
    try:
        # --- 1. 获取文件大小 ---
        file_size_bytes = os.path.getsize(file_path)
        file_size_mb = file_size_bytes / (1024 * 1024)
        
        # --- 2. 加载文件内容 ---
        data = np.load(file_path, allow_pickle=True)
        
        # --- 3. 确定样本数量 ---
        # 样本总数 (N_samples) 存储在 num_atoms 字段的第一维
        if 'reactant/num_atoms' in data:
            num_samples = data['reactant/num_atoms'].shape[0]
            
            # 确定最大原子数 (Padding 长度)
            if 'reactant/positions' in data:
                max_atoms = data['reactant/positions'].shape[1]
            else:
                max_atoms = "未知"
                
            print("\n==================================================")
            print(f"文件分析: {file_path.name}")
            print("==================================================")
            print(f"📂 文件大小: {file_size_mb:.2f} MB")
            print(f"📊 反应路径总数 (样本数): {num_samples} 个反应")
            print(f"📐 每个反应的最大原子数 (Padding): {max_atoms}")
            print("\n--- 关键字段形状 (Shape) ---")
            print(f" - Reactant Positions: {data['reactant/positions'].shape}")
            print(f" - Reactant Num Atoms: {data['reactant/num_atoms'].shape}")
            
        else:
            print("错误: NPZ 文件结构不完整，缺少 'reactant/num_atoms' 键。")
            
    except FileNotFoundError:
        print(f"\n错误: 未找到文件 {file_path}。请检查路径是否正确。")
    except Exception as e:
        print(f"\n加载或分析文件时发生错误: {e}")

if __name__ == "__main__":
    analyze_custom_npz(NPZ_FILE_PATH)


文件分析: custom_R_P_multi_molecule.npz
📂 文件大小: 161.17 MB
📊 反应路径总数 (样本数): 18694 个反应
📐 每个反应的最大原子数 (Padding): 150

--- 关键字段形状 (Shape) ---
 - Reactant Positions: (18694, 150, 3)
 - Reactant Num Atoms: (18694,)


In [11]:
'''
import copy
import numpy as np
import torch
from torch.utils.data import DataLoader

from dataset.base_dataset import BaseDataset, ATOM_MAPPING

ATOM_MAPPING = {
    1: 0,   # 氢 (H)
    5: 1,   # 硼 (B)
    6: 2,   # 碳 (C)
    7: 3,   # 氮 (N)
    8: 4,   # 氧 (O)
    9: 5,   # 氟 (F)
    15: 6,  # 磷 (P)
    16: 7,  # 硫 (S)
    17: 8,  # 氯 (Cl)
    35: 9,  # 溴 (Br)
    53: 10, # 碘 (I)
}

n_element = len(list(ATOM_MAPPING.keys()))
FRAG_MAPPING = {
    "reactant": "product",
    "transition_state": "transition_state",
    "product": "reactant",
}


class ProcessedTS1x(BaseDataset):
    
    def _restructure_npz_data(self):
        """将扁平的 'fragment/key' 结构重构为 ProcessedTS1x 期望的嵌套字典结构。"""
        raw_data = self.raw_dataset
        nested_data = {}
        
        # 1. 重构嵌套结构
        for flat_key, array in raw_data.items():
            if '/' in flat_key:
                # 分割键名: 'reactant/positions' -> 'reactant', 'positions'
                fragment, key = flat_key.split('/', 1)
                
                if fragment not in nested_data:
                    nested_data[fragment] = {}
                
                # 存储数据到嵌套字典
                nested_data[fragment][key] = array
            else:
                # 保留顶层元数据（如 single_fragment, use_ind）
                nested_data[flat_key] = array
        
        # 2. 替换原始数据
        self.raw_dataset = nested_data
        
        
    def __init__(
        self,
        npz_path,
        center=True,
        pad_fragments=0,
        device="cpu",
        zero_charge=False,
        remove_h=False,
        single_frag_only=True,
        swapping_react_prod=False,
        append_frag=False,
        reflection=False,
        use_by_ind=True,
        only_ts=False,
        confidence_model=False,
        position_key="positions",
        ediff=None,
        **kwargs,
    ):
        super().__init__(
            npz_path=npz_path,
            center=center,
            device=device,
            zero_charge=zero_charge,
            remove_h=remove_h,
        )
        
        # <<< FIX START >>>
        # 检查并重构数据结构，如果它是来自 NPZ 文件的扁平结构
        if ".npz" in str(npz_path):
             self._restructure_npz_data()
        # <<< FIX END >>>

        # (其余代码保持不变)
        if confidence_model:
            use_by_ind = False
        # ... (此处省略其余的 ProcessedTS1x.__init__ 逻辑)
        
        # 由于我们只关心修复 KeyError，这里只保留关键的逻辑块
        if single_frag_only:
            # ... (假设 single_frag_inds 已经被正确计算)
            single_frag_inds = np.array(range(len(self.raw_dataset["single_fragment"]))) 
        else:
            single_frag_inds = np.array(range(len(self.raw_dataset["single_fragment"])))
            
        if use_by_ind:
            # ... (use_inds 逻辑)
            use_inds = range(len(self.raw_dataset["single_fragment"])) 
        else:
            use_inds = range(len(self.raw_dataset["single_fragment"]))
            
        single_frag_inds = list(set(single_frag_inds).intersection(set(use_inds)))


        data_duplicated = copy.deepcopy(self.raw_dataset)
        
        # 错误发生的循环，现在可以正常访问 data_duplicated['reactant']
        for k, mapped_k in FRAG_MAPPING.items():
             # 这行现在应该能正常工作: data_duplicated['reactant'].items()
            for v, val in data_duplicated[k].items(): 
                self.raw_dataset[k][v] = [val[ii] for ii in single_frag_inds]
                if swapping_react_prod:
                    mapped_val = data_duplicated[mapped_k][v]
                    self.raw_dataset[k][v] += [
                        mapped_val[ii] for ii in single_frag_inds
                    ]
                    
        # ... (省略其余初始化代码)
        self.reactant = self.raw_dataset["reactant"]
        self.transition_state = self.raw_dataset["transition_state"]
        self.product = self.raw_dataset["product"]

        self.n_fragments = pad_fragments + 3
        self.device = torch.device(device)
        self.n_samples = len(self.reactant["charges"])

        self.data = {}
        repeat = 2 if swapping_react_prod else 1
        
        # 修复逻辑只需要保证 self.raw_dataset 在这里是嵌套的即可。
        # 假设 ProcessedTS1x 的其余部分逻辑不变。
        if not only_ts:
            if not append_frag:
                self.process_molecules(
                    "reactant", self.n_samples, idx=0, position_key=position_key
                )
                self.process_molecules("transition_state", self.n_samples, idx=1)
                self.process_molecules(
                    "product", self.n_samples, idx=2, position_key=position_key
                )
            # ... (省略其余逻辑)
            
        self.data["condition"] = [
            torch.zeros(
                size=(1, 1),
                dtype=torch.int64,
                device=self.device,
            )
            for _ in range(self.n_samples)
        ]
'''

'\nimport copy\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom dataset.base_dataset import BaseDataset, ATOM_MAPPING\n\nATOM_MAPPING = {\n    1: 0,   # 氢 (H)\n    5: 1,   # 硼 (B)\n    6: 2,   # 碳 (C)\n    7: 3,   # 氮 (N)\n    8: 4,   # 氧 (O)\n    9: 5,   # 氟 (F)\n    15: 6,  # 磷 (P)\n    16: 7,  # 硫 (S)\n    17: 8,  # 氯 (Cl)\n    35: 9,  # 溴 (Br)\n    53: 10, # 碘 (I)\n}\n\nn_element = len(list(ATOM_MAPPING.keys()))\nFRAG_MAPPING = {\n    "reactant": "product",\n    "transition_state": "transition_state",\n    "product": "reactant",\n}\n\n\nclass ProcessedTS1x(BaseDataset):\n\n    def _restructure_npz_data(self):\n        """将扁平的 \'fragment/key\' 结构重构为 ProcessedTS1x 期望的嵌套字典结构。"""\n        raw_data = self.raw_dataset\n        nested_data = {}\n\n        # 1. 重构嵌套结构\n        for flat_key, array in raw_data.items():\n            if \'/\' in flat_key:\n                # 分割键名: \'reactant/positions\' -> \'reactant\', \'positions\'\n                fragment, 

In [12]:
import copy
import numpy as np
import torch
from torch.utils.data import DataLoader

from oa_reactdiff.dataset.base_dataset import BaseDataset, ATOM_MAPPING

device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda")
print("Using device:", device)

Using device: cuda


In [13]:
from oa_reactdiff.dataset.transition1x import ProcessedTS1x

dataset = ProcessedTS1x(
    npz_path="./oa_reactdiff/data_meci/custom_R_P_multi_molecule.npz",
    center=True,
    pad_fragments=0,
    device=device,
    zero_charge=False,
    remove_h=False,
    single_frag_only=False,
    swapping_react_prod=False,
    use_by_ind=True,
)


[DEBUG CHECK] ATOM_MAPPING KEYS IN USE: [1, 5, 6, 7, 8, 9, 15, 16, 17, 35, 53]
[DEBUG CHECK] n_element (Expected: 11): 11

[DEBUG CHECK] ATOM_MAPPING KEYS IN USE: [1, 5, 6, 7, 8, 9, 15, 16, 17, 35, 53]
[DEBUG CHECK] n_element (Expected: 11): 11

[DEBUG CHECK] ATOM_MAPPING KEYS IN USE: [1, 5, 6, 7, 8, 9, 15, 16, 17, 35, 53]
[DEBUG CHECK] n_element (Expected: 11): 11


In [14]:
# 假设您已定义了 dataset 对象
# dataset = ProcessedTS1x(...)
BATCH_SIZE = 3

loader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=dataset.collate_fn # 使用自定义的 collate_fn
)

# 即可开始遍历批次数据
itl = iter(loader)
first_batch = next(itl)
# first_batch[0] 是 [Reactant, TS, Product] 的列表

In [15]:
(out, conditions) = next(itl)
#print(out,conditions)
print("\n--- 反应物 (Reactant) 数据结构：out[0] ---")
reactant_batch = out[0]
for key, tensor in reactant_batch.items():
    print(f"键: {key:<10} | 形状: {tensor.shape} | 描述: {key.upper()}")

# 1. 查看 Reactant 的位置 (pos)
print("\n[Reactant 结构详情 - 位置 (pos)]")
pos_r = reactant_batch['pos']
print(f"形状: {pos_r.shape}")
print(f"总原子数 (N_total): {pos_r.shape[0]}. 这是 {BATCH_SIZE} 个反应物的原子总数。")
# 4 (Sample 0) + 6 (Sample 1) = 10，所以形状应该是 (10, 3)

# 2. 查看 Reactant 的批次索引 (mask)
print("\n[Reactant 结构详情 - 批次索引 (mask)]")
mask_r = reactant_batch['mask']
print(f"形状: {mask_r.shape}")
print(f"前10个 mask 值: {mask_r[:]}")
# 预期值：[0, 0, 0, 0, 1, 1, 1, 1, 1, 1] (4个0代表第一个样本，6个1代表第二个样本)

# 3. 查看 Reactant 的电荷/片段 ID (charge)
print("\n[Reactant 结构详情 - 电荷/片段 ID (charge)]")
charge_r = reactant_batch['charge']
print(f"形状: {charge_r.shape}") 
# 形状 (N_total, 2)，因为我们设置了 append_frag=True。第二列是片段ID。
print(f"片段 ID 维度 (最后一列) 的前10个值: {charge_r[:10, -1]}")
# 预期值：全部为 0 (因为 Reactant 传入的 append_charge=0)

print("\n======================================================\n")

print("--- 生成物 (Product) 数据结构：out[2] ---")
product_batch = out[2]
for key, tensor in product_batch.items():
    print(f"键: {key:<10} | 形状: {tensor.shape} | 描述: {key.upper()}")

# 4. 查看 Product 的位置 (pos)
pos_p = product_batch['pos']
print("\n[Product 结构详情 - 位置 (pos)]")
print(f"形状: {pos_p.shape}")
# 形状与 Reactant 相同：(10, 3)

# 5. 查看 Product 的电荷/片段 ID (charge)
print("\n[Product 结构详情 - 电荷/片段 ID (charge)]")
charge_p = product_batch['charge']
print(f"形状: {charge_p.shape}") 
print(f"片段 ID 维度 (最后一列) 的前10个值: {charge_p[:10, -1]}")
# 预期值：全部为 0 (因为 Product 传入的 append_charge=0)

print("\n======================================================\n")

print("--- 过渡态 (Transition State) 数据结构：out[1] ---")
ts_batch = out[1]
for key, tensor in ts_batch.items():
    # 使用 f-string 格式化，确保输出整齐
    print(f"键: {key:<10} | 形状: {tensor.shape} | 描述: {key.upper()}")

# 4. 查看 Transition State 的位置 (pos)
pos_ts = ts_batch['pos']
print("\n[Transition State 结构详情 - 位置 (pos)]")
print(f"形状: {pos_ts.shape}")
# 形状与 Reactant/Product 相同：(N_total, 3)。N_total是5个样本的原子总数。

# 5. 查看 Transition State 的电荷/片段 ID (charge)
charge_ts = ts_batch['charge']
print("\n[Transition State 结构详情 - 电荷/片段 ID (charge)]")
print(f"形状: {charge_ts.shape}") 
# 形状 (N_total, D)，D通常为1（默认）或2（如果设置了 append_frag=True）

# 打印最后一个维度，用于检查片段 ID
print(f"片段 ID 维度 (最后一列) 的前10个值: {charge_ts[:10, -1]}")

# 预期值分析：
# 如果 ProcessedTS1x 初始化时设置了 append_frag=True，
# 那么 charge 向量的最后一列将全部为 1，因为 1 是过渡态的标记。
# 如果是默认配置（append_frag=False），则这一步会打印原始电荷（原子序数）的最后一维，形状是 (N_total, 1)。


--- 反应物 (Reactant) 数据结构：out[0] ---
键: size       | 形状: torch.Size([3]) | 描述: SIZE
键: pos        | 形状: torch.Size([70, 3]) | 描述: POS
键: one_hot    | 形状: torch.Size([70, 11]) | 描述: ONE_HOT
键: charge     | 形状: torch.Size([70, 1]) | 描述: CHARGE
键: mask       | 形状: torch.Size([70]) | 描述: MASK

[Reactant 结构详情 - 位置 (pos)]
形状: torch.Size([70, 3])
总原子数 (N_total): 70. 这是 3 个反应物的原子总数。

[Reactant 结构详情 - 批次索引 (mask)]
形状: torch.Size([70])
前10个 mask 值: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
       device='cuda:0')

[Reactant 结构详情 - 电荷/片段 ID (charge)]
形状: torch.Size([70, 1])
片段 ID 维度 (最后一列) 的前10个值: tensor([7, 6, 8, 6, 6, 6, 6, 6, 6, 7], device='cuda:0')


--- 生成物 (Product) 数据结构：out[2] ---
键: size       | 形状: torch.Size([3]) | 描述: SIZE
键: pos        | 形状: torch.Size([70, 3]) | 描述: POS
键: one_hot    | 形状: torch